# 10 · Dark current and CIC: the joint fit

*Level: advanced. Guide Section 14. Code: `structured.JointLambdaGP`, `structured.line_fit_covariance`, `src/image_rows.py`.*

**Idea.** The dark charge λ has two sources:
* **dark current**, which grows with the time a pixel waits;
* **clock-induced charge (CIC)**, from the clock swings.

Rows read later wait longer, so the image separates the two. Their estimates are strongly **anti-correlated**, and a model must use that.

In [ ]:
from tutorial_utils import *

from ml import structured as st
import image_rows as ir

## 1. λ row by row

$$\lambda_r = \mathrm{CIC} + \mathrm{DC}\,x_r,\qquad x_r = \frac{r+\tfrac12}{R/2}.$$

Each row's λ is fitted from its pixels by maximum likelihood (`image_rows.row_lambda`), with the gain and noise fixed. Here it is tried on
simulated pixels.

In [ ]:
rng = np.random.default_rng(0); R, C, g, s, dc, cic = 16, 844, 1.3, 0.25, 0.02, 0.01
xr = (np.arange(R) + 0.5) / (R / 2); lam = cic + dc * xr
img = np.array([100 + g * rng.poisson(l, C) + s * rng.normal(size=C) for l in lam])
plt.figure(figsize=(5, 3)); plt.hist(img[8] - 100, 120); plt.yscale("log"); plt.xlabel("pixel value − pedestal (ADU)"); plt.title("one row: 0-, 1-, 2-electron peaks"); plt.show()
fit = np.array([ir.row_lambda(row, g, s) for row in img])
print("row 0: λ = %.4f ± %.4f (truth %.4f)" % (*fit[0], lam[0]))

## 2. The straight line and its covariance, by hand

With $A$ the $R\times2$ matrix of rows $(x_r, 1)$, least squares gives

$$\begin{pmatrix}\widehat{\mathrm{DC}}\\ \widehat{\mathrm{CIC}}\end{pmatrix} = P\boldsymbol\lambda,\quad P = (A^\top A)^{-1}A^\top,\qquad \Sigma = P\,\mathrm{diag}(\sigma_r^2)\,P^\top .$$

Repeating the simulated image 2000 times shows the anti-correlation.

In [ ]:
A = np.column_stack([xr, np.ones(R)]); P = np.linalg.solve(A.T @ A, A.T)
sig_r = 0.12 * lam                                           # each row's λ to ~12 %
Sigma = P @ np.diag(sig_r ** 2) @ P.T
draws = np.array([P @ (lam + sig_r * rng.normal(size=R)) for _ in range(2000)])
print("covariance by formula (×1e6):\n", (Sigma * 1e6).round(3), "\ncorrelation:", (Sigma[0, 1] / np.sqrt(Sigma[0, 0] * Sigma[1, 1])).round(3))
print("library, structured.line_fit_covariance (×1e6):\n", (st.line_fit_covariance(np.array([dc]), np.array([cic]))[0] * 1e6).round(3))
plt.figure(figsize=(4.5, 3.5)); plt.scatter(*draws.T, s=2, alpha=0.3); plt.xlabel("DC estimate"); plt.ylabel("CIC estimate"); plt.show()
var_sum = Sigma[0, 0] + Sigma[1, 1] + 2 * Sigma[0, 1]
print(f"sd of DC + CIC: with the covariance {np.sqrt(var_sum):.5f}; ignoring it {np.sqrt(Sigma[0, 0] + Sigma[1, 1]):.5f}  (much worse)")

**Fitting the two parts separately** throws this correlation away: each part looks uncertain, while their sum (the image's mean λ) is precise.

**The joint model** has two GPs, $f_\text{DC}$ and $f_\text{CIC}$, observed together with each image's known $2\times2$ covariance. In logs
the covariance is converted by the delta method, $\Sigma_\text{log} = J\Sigma J$ with $J = \mathrm{diag}(1/\mathrm{DC}, 1/\mathrm{CIC})$.

## 3. On the toy CCD: direct, separate, and joint λ

The toy world v3 has per-row λ and an image-only noise term. Errors are against the truth.

In [ ]:
kw = dict(per_row_lambda=True, image_only_noise=True)
dft, ct = synthetic.make_dataset(60, seed=2, **kw)
truth_t = np.log(dft[["truth.sigma_e", "truth.lambda", "truth.gain"]].to_numpy())
for n in (60, 120):
    dtr, ctr = synthetic.make_dataset(n, seed=1, **kw); Ytr = data.make_targets(dtr)[0]
    prep = st.StructuredPrep().fit(dtr); Xtr, Xte = prep.transform(dtr, ctr), prep.transform(dft, ct)
    for label, model, comp in (("direct GP on log λ", st.PhysicsStructured(), None),
                               ("separate DC and CIC GPs", st.PhysicsStructured(use_components=True), st.component_targets(dtr)),
                               ("joint DC + CIC (with covariance)", st.PhysicsStructured(lambda_model="joint"), None)):
        m, s = model.fit(Xtr, Ytr, comp).predict(Xte)
        print(f"{n:3d} packets | {label:34s}: λ error {100 * np.sqrt(np.mean((m[:, 1] - truth_t[:, 1]) ** 2)):5.1f} %")

## When to use it

**Reading the numbers:** the joint fit beats fitting the two parts separately. Against the direct fit it varies: in this single
run, without amplifier labels, the direct fit is ahead. Over 3 training scans in the project's test (with the hierarchical GP) they were equal
within the scatter (1.6 against 1.8 % at 240 packets).

* **Good:** when λ row by row is available and you want dark current and CIC **separately** (which settings drive which). For the total λ the
  direct fit is as good or better.
* **Avoid:** fitting the parts with separate models.

Next: **11 · Calibrated error bars (conformal prediction).**